# Riverside Live Test

This notebook is a tiny watch-me-run version.

It does one full test with visible output in each step.

## Cell 1: Setup

This cell finds the project folder, makes sure notebook code can import the chatbot files, and checks whether the embedding package is installed.

If the package is missing, this cell installs it automatically in the same Python environment the notebook is using.

In [1]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "faqs.json").exists():
            return candidate
    raise FileNotFoundError("Could not find project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

EMBEDDINGS_AVAILABLE = importlib.util.find_spec("sentence_transformers") is not None

print("Project root found:")
print(PROJECT_ROOT)
print()
print("Notebook Python executable:")
print(sys.executable)
print()
if EMBEDDINGS_AVAILABLE:
    print("Embedding package status:")
    print("sentence_transformers is installed. The semantic matcher can run.")
else:
    print("Embedding package status:")
    print("sentence_transformers is NOT installed in this notebook environment.")
    print()
    print("Installing it now in this notebook environment...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sentence-transformers"])
    EMBEDDINGS_AVAILABLE = importlib.util.find_spec("sentence_transformers") is not None
    print()
    if EMBEDDINGS_AVAILABLE:
        print("Install complete. The semantic matcher can run now.")
    else:
        print("Install attempt finished, but sentence_transformers still was not found.")
        print("If needed, run this manually:")
        print(f"{sys.executable} -m pip install sentence-transformers")

Project root found:
c:\Users\sossi\OneDrive\Desktop\agent builder workspace\Riverside Chatbot solution

Notebook Python executable:
c:\Users\sossi\AppData\Local\Programs\Python\Python311\python.exe

Embedding package status:
sentence_transformers is installed. The semantic matcher can run.


## Cell 2: Load the FAQs

This cell opens the FAQ file and shows a small sample.

In [2]:
from src.data import load_faqs

faqs = load_faqs()
print(f"Loaded {len(faqs)} FAQs")
print("First FAQ question:")
print(faqs[0].question)
print("First FAQ answer:")
print(faqs[0].answer)

Loaded 20 FAQs
First FAQ question:
What are your opening hours?
First FAQ answer:
We're open 9am to 6pm Monday to Saturday, and 11am to 4pm on Sundays.


## Cell 3: Create the matcher

This cell builds the smarter matcher and turns debug mode on so we can see what it is thinking.

It also reminds you whether embeddings are available or whether the chatbot will fall back to the simpler word-matching mode.

In [3]:
from src.matchers import SemanticMatcher, _prepare_query

matcher = SemanticMatcher(debug=True, logger=print)
print("Matcher created:")
print(type(matcher).__name__)
print()
if EMBEDDINGS_AVAILABLE:
    print("Mode:")
    print("Semantic matcher is ready. It should use embeddings.")
    print("Loading the embedding model now so the later match cell stays cleaner...")
    matcher._load_model()
    matcher._ensure_embeddings(faqs)
    print("Embedding model loaded and FAQ embeddings are ready.")
else:
    print("Mode:")
    print("Embeddings are missing, so the chatbot will fall back to lexical matching.")

Matcher created:
SemanticMatcher

Mode:
Semantic matcher is ready. It should use embeddings.
Loading the embedding model now so the later match cell stays cleaner...


c:\Users\sossi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10247.97it/s]


Embedding model loaded and FAQ embeddings are ready.


## Cell 3.5: Helper tools for neat output

This cell creates two small helper functions:
- one prints a clean rank table
- one runs a question and prints the full result nicely

That way we can reuse the same neat output for the sample question and for your own typed question.

In [4]:
def print_rank_table(result):
    print("Rank table:")
    print(f"{'Rank':<6} {'FAQ ID':<8} {'Score':<8} Question")
    print("-" * 80)
    for index, candidate in enumerate(result.candidates, start=1):
        print(f"{index:<6} {candidate.faq.id:<8} {candidate.score:<8.3f} {candidate.faq.question}")


def show_question_result(question: str):
    print("User question:")
    print(question)
    print("Prepared version used by the matcher:")
    print(_prepare_query(question))
    print()
    result = matcher.match(question, faqs)

    if result is None:
        print("No safe match was found.")
        return None

    print_rank_table(result)
    print()
    print("Winning FAQ id:")
    print(result.faq.id)
    print("Winning FAQ question:")
    print(result.faq.question)
    print("Answer shown to the user:")
    print(result.faq.answer)
    print("Final score:")
    print(round(result.score, 3))
    print("Gap over second place:")
    print(round(result.margin, 3))
    return result

## Cell 4: Pick one real customer question

We will test this question:

`where's the shop?`

In [5]:
query = input("Type your Riverside Books question here: ").strip()
print("Your question:")
print(query)

Your question:



## Cell 5: Run the match

This is the real test. The matcher should choose the location FAQ.

In [6]:
result = matcher.match(query, faqs)



# Always show candidates by temporarily disabling thresholds

original_min_score = matcher.min_score

original_min_margin = matcher.min_margin

matcher.min_score = 0.0

matcher.min_margin = 0.0

full_result = matcher.match(query, faqs)

matcher.min_score = original_min_score

matcher.min_margin = original_min_margin



if full_result is None:

    print("No candidates found.")

else:

    print("Rank table (closest matches first):")

    print(f"{'Rank':<6} {'FAQ ID':<8} {'Score':<8} Question")

    print("-" * 80)

    for index, candidate in enumerate(full_result.candidates, start=1):

        print(f"{index:<6} {candidate.faq.id:<8} {candidate.score:<8.3f} {candidate.faq.question}")

    print()



    if result is None:

        print("No safe match was found (score or margin below threshold).")

    else:

        print("Winning FAQ id:")

        print(result.faq.id)

        print("Winning FAQ question:")

        print(result.faq.question)

        print("Answer shown to the user:")

        print(result.faq.answer)

        print("Final score:")

        print(round(result.score, 3))

        print("Gap over second place:")

        print(round(result.margin, 3))

Debug top candidates -> 6:0.258 Do you host events? | 20:0.224 Do you offer student discounts? | 12:0.216 Do you offer click and collect?
Debug top candidates -> 6:0.258 Do you host events? | 20:0.224 Do you offer student discounts? | 12:0.216 Do you offer click and collect?
Rank table (closest matches first):
Rank   FAQ ID   Score    Question
--------------------------------------------------------------------------------
1      6        0.258    Do you host events?
2      20       0.224    Do you offer student discounts?
3      12       0.216    Do you offer click and collect?

No safe match was found (score or margin below threshold).


## Cell 6: Final plain-English takeaway

This final cell explains the result in simple words.

In [ ]:
if result is None:

    print("Sorry I cannot confidently answer your question. Shall I connect you to a member of staff?")


else:

    print(result.faq.answer)    

The chatbot decided it was safer to say 'I don't know' than to guess.

Sorry I cannot confidently answer your question. Shall I connect you to a member of staff?
Yes, we run author readings and a monthly book club. Check our noticeboard for dates.
